In [1]:
import numpy as np
import matplotlib.pyplot as plt 
import pandas as pd
from scipy.optimize import brentq
from solvers.background import solve_tov, eos, real_eos,eos_kokkotas
from solvers.shooting_gr import get_mode,solve_system
from scipy.interpolate import interp1d

from IPython.display import display, Markdown
plt.style.use('seaborn-v0_8-whitegrid')

In [2]:
# Θεμελιώδεις Σταθερές & Παράμετροι Μοντέλου
c = 300000
M_sun = 1.4766            # Μάζα Ήλιου σε km (γεωμετρικές μονάδες)
rho_cgs_to_geom = 7.4e-19 # Μετατροπή από g/cm^3 σε km^-2
gamma = 2                 

In [3]:
#Αυτό για την πολυτροπική eos
rho_c_values = [1.00, 1.50, 2.00, 3.00, 4.00 , 5.00, 5.30, 5.50, 5.60,5.65,5.70]

Pc_values = []
for rhoc in rho_c_values:
    rho_c_geom = rhoc*1e15*rho_cgs_to_geom
    Pc = ((2*rho_c_geom + 1/100) - np.sqrt(4*rho_c_geom/100 + (1/100)**2))/2
    Pc_values.append(Pc)

    print(rf'{Pc:.3e}')

4.790e-05
1.017e-04
1.713e-04
3.498e-04
5.708e-04
8.260e-04
9.083e-04
9.644e-04
9.929e-04
1.007e-03
1.022e-03


In [4]:
#Ρεαλιστική EoS όπως στο paper των OV
t = np.linspace(1e-5,5,1000)
rho_real, P_real = real_eos(t)

eos_real = interp1d(P_real,rho_real, kind="cubic", fill_value="extrapolate")
inverse_eos_real = interp1d(rho_real, P_real, kind="cubic", fill_value="extrapolate")
dP_drho = np.gradient(P_real, rho_real)
cs2_interp = interp1d(P_real, dP_drho, kind="cubic", fill_value="extrapolate")

In [187]:
# 1. Επιλογή "στρογγυλών" τιμών ρ_c μέχρι την κρίσιμη πυκνότητα (4.34)
rho_c_targets = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.34])
rho_geom_targets = (rho_c_targets * 1e15) * rho_cgs_to_geom

# 2. Εύρεση των Pc μέσω της συνάρτησης παρεμβολής
Pc_values = inverse_eos_real(rho_geom_targets)
for Pc in Pc_values:
    print(rf'{Pc:.3e}')

1.202e-05
3.542e-05
6.572e-05
1.011e-04
1.407e-04
1.836e-04
2.295e-04
2.780e-04
3.122e-04


In [219]:
Pc = 3.122*1e-4
rho_c = eos_real(Pc)
#rho_c_15 = (rho_c*rho_cgs_to_geom) / 1e15

R_tov, M_tov,Phi_tov_offset, sol = solve_tov(Pc, EoS=eos_real)
display(Markdown(rf"Η ακτίνα του άστρου είναι **{R_tov:.3f}** km"))
display(Markdown(rf"Η μάζα του άστρου είναι **{M_tov/1.4766:.3f}** $M_\odot$"))

r = np.linspace(1e-4,R_tov*0.999,200)
m = sol.sol(r)[0]
P_0 = sol.sol(r)[1]
Phi = sol.sol(r)[2]
rho_0 = eos_real(P_0)

L_0 = (-1/2)*np.log(1-(2*m/r)) #Lambda
Phi_0 = Phi + Phi_tov_offset

cs2_local = cs2_interp(np.maximum(P_0, 0.0))

#Ορίζω τις P,W,Q συναρτήσεις, με βάση το μοντέλο ισσοροπίας που έχω (πολυτροπο Γ=2)
Gamma1P_0 = (rho_0 + P_0) * cs2_local
P_fun = (Gamma1P_0/r**2)*np.exp(L_0+3*Phi_0)
W_fun = ((rho_0+P_0)/r**2)*np.exp(3*L_0 + Phi_0)
dPhi_dr = (-1/(2*r))*(1-np.exp(2*L_0)) + 4*np.pi * r * P_0 * np.exp(2*L_0)
Q_fun = (np.exp(L_0+3*Phi_0))/(r**2)*(rho_0 + P_0)*((dPhi_dr**2) + (4/r)*dPhi_dr -8*np.pi*np.exp(2*L_0)*P_0)

P = interp1d(r, P_fun, "linear", fill_value="extrapolate")
W = interp1d(r, W_fun, "linear", fill_value="extrapolate")
Q = interp1d(r, Q_fun, "linear", fill_value="extrapolate")  

Η ακτίνα του άστρου είναι **9.087** km

Η μάζα του άστρου είναι **0.710** $M_\odot$

In [220]:
from solvers.shooting_gr import solve_system,get_mode 

omega_scan = np.linspace(0.000001,0.01,40)
margin = 1e-2
for w2 in omega_scan:
    try:
        residual, _ = solve_system(r,P,W,Q,omega2=w2)
        if (residual-margin)*(residual+margin) < 0: 
            print(f"Βρέθηκε υποψήφια ιδιοσυχνότητα κοντά στο: {w2}")
            break
    except Exception as e:
        print(f"Σφάλμα στο w2={w2}: {e}")
        break

Βρέθηκε υποψήφια ιδιοσυχνότητα κοντά στο: 1e-06


In [222]:
eig_freq = get_mode(0.0000001,0.0006,r,P,W,Q)
#Σε μονάδες kHz 
fHz = (np.sqrt(eig_freq)*c)/(1000*2*np.pi)
display(Markdown(rf"Η συχνότητα του πρώτου f-mode για $\rho_c = $ **{eos_real(P_0[0])/(rho_cgs_to_geom):2e}** $g/cm^3$ είναι **{fHz:.3f}** $kHz$"))

Αποτυχία εύρεσης ρίζας στο διάστημα [1e-07, 0.0006].
Σφάλμα: f(a) and f(b) must have different signs


TypeError: loop of ufunc does not support argument 0 of type NoneType which has no callable sqrt method

In [223]:
eig_freq = get_mode(-0.01,1e-8,r,P,W,Q)
fHz = (np.sqrt(-eig_freq)*c)/(1000)
e_fold = 1/fHz

In [224]:
e_fold

0.5522928318647878